# 02 — Data Preparation

## Objective
Transform raw Favorita data into a clean, feature-engineered star schema
ready for both Azure AutoML training and Power BI visualization.

**What this notebook does:**
1. Runs the data preparation pipeline (`src/data_prep.py`)
2. Validates the output tables
3. Visualizes the star schema structure
4. Quality-checks the feature engineering

This notebook calls the production scripts directly — so the same code
that prepares data here is what runs in the actual pipeline.

In [ ]:
# === Setup ===
import sys
sys.path.insert(0, '..')  # Allow imports from project root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

PROCESSED_DIR = Path('../data/processed')
print('Ready to prepare data.')

## 1. Run the Data Pipeline

The `data_prep.py` script handles:
- Downloading from Kaggle (if needed)
- Cleaning and merging all 5 source tables
- Engineering calendar, economic, and promotion features
- Building the star schema (dim_date, dim_store, dim_product, fact_sales)
- Exporting to CSV and Parquet

In [ ]:
# Run the pipeline script
# --skip-download if you already have the data in data/raw/
!cd .. && python src/data_prep.py --skip-download

## 2. Validate Star Schema Tables

Let's verify each table was created correctly with the expected schema.

In [ ]:
# Load all star schema tables
dim_date = pd.read_csv(PROCESSED_DIR / 'dim_date.csv', parse_dates=['date'])
dim_store = pd.read_csv(PROCESSED_DIR / 'dim_store.csv')
dim_product = pd.read_csv(PROCESSED_DIR / 'dim_product.csv')
fact_sales = pd.read_csv(PROCESSED_DIR / 'fact_sales.csv')

# Summary of each table
tables = {
    'dim_date': dim_date,
    'dim_store': dim_store, 
    'dim_product': dim_product,
    'fact_sales': fact_sales,
}

print(f'{"Table":<15} {"Rows":>10} {"Columns":>10} {"Null %":>10}')
print('-' * 50)
for name, df in tables.items():
    null_pct = df.isnull().mean().mean() * 100
    print(f'{name:<15} {len(df):>10,} {len(df.columns):>10} {null_pct:>9.1f}%')

In [ ]:
# Validate referential integrity
# Every foreign key in fact_sales should exist in the dimension tables

# Check date keys
fact_dates = set(fact_sales['date_key'].unique())
dim_dates = set(dim_date['date_key'].unique())
orphan_dates = fact_dates - dim_dates
print(f'Date key integrity: {"✓ PASS" if len(orphan_dates) == 0 else f"✗ {len(orphan_dates)} orphans"}')

# Check store keys
fact_stores = set(fact_sales['store_key'].unique())
dim_stores = set(dim_store['store_key'].unique())
orphan_stores = fact_stores - dim_stores
print(f'Store key integrity: {"✓ PASS" if len(orphan_stores) == 0 else f"✗ {len(orphan_stores)} orphans"}')

# Check product keys
fact_products = set(fact_sales['product_key'].unique())
dim_products = set(dim_product['product_key'].unique())
orphan_products = fact_products - dim_products
print(f'Product key integrity: {"✓ PASS" if len(orphan_products) == 0 else f"✗ {len(orphan_products)} orphans"}')

In [ ]:
# Preview each table
print('=== dim_date (first 5 rows) ===')
display(dim_date.head())

print('\n=== dim_store ===')
display(dim_store.head())

print('\n=== dim_product ===')
display(dim_product.head())

print('\n=== fact_sales (first 5 rows) ===')
display(fact_sales.head())

## 3. Star Schema Diagram

```
┌─────────────────────┐         ┌────────────────────────────────────┐
│     dim_date        │         │          fact_sales                │
├─────────────────────┤    ┌──→ ├────────────────────────────────────┤
│ date_key (PK)       │────┘    │ date_key (FK)                     │
│ date                │         │ store_key (FK)                    │
│ year, month, quarter│         │ product_key (FK)                  │
│ day_of_week         │         │ sales, revenue, margin            │
│ is_weekend          │         │ onpromotion, oil_price            │
│ is_holiday          │         │ is_holiday, transactions          │
└─────────────────────┘         └────────────────────────────────────┘
                                        ▲              ▲
┌─────────────────────┐                │              │
│    dim_store         │────────────────┘              │
├─────────────────────┤                               │
│ store_key (PK)      │    ┌──────────────────────┐   │
│ store_nbr           │    │   dim_product         │───┘
│ city, state, region │    ├──────────────────────┤
│ store_type, cluster │    │ product_key (PK)     │
└─────────────────────┘    │ family, category     │
                           └──────────────────────┘
```

In [ ]:
# Revenue distribution by product category
# This shows how the synthetic revenue distributes across categories
product_revenue = (
    fact_sales.merge(dim_product, on='product_key')
    .groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
product_revenue.plot(kind='barh', ax=ax, color='#2563EB', alpha=0.8)
ax.set_title('Total Revenue by Product Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue ($)')
plt.tight_layout()
plt.show()

## 4. Summary

**Data preparation complete.** The star schema is ready for:
- **Azure AutoML**: `automl_training_data.parquet` (flat, feature-rich)
- **Power BI**: `dim_date.csv`, `dim_store.csv`, `dim_product.csv`, `fact_sales.csv`

**Next step:** `03_automl_training.ipynb` — train the forecasting model.